# Task 7 - export the governed model for Power BI

The brief requires Power BI to consume a **governed analytical model**, not a
flat multi-table visual model. This notebook writes a proper star schema:
one slim fact at transaction-attempt grain, five conformed dimensions, and two
pre-computed analysis tables that DAX cannot reproduce cleanly.

The full fact table has 45 columns and is 88 MB. Power BI does not need most of
them, so the export is trimmed to the columns the report actually uses.

Put this notebook in `07_PowerBI/`.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

CANDIDATES = [
    Path("model_output/fact_transaction.csv"),
    Path("../04_profitability_model/model_output/fact_transaction.csv"),
    Path("../05_profitability_model/model_output/fact_transaction.csv"),
]
FACT = next((p for p in CANDIDATES if p.exists()), None)
if FACT is None:
    raise FileNotFoundError("Run the Task 4 notebook first")

DATA_DIR = next((p for p in [Path("../data"), Path("data"), Path("../../data")]
                 if (p / "route_cost.csv").exists()), None)

OUT = Path("powerbi_model")
OUT.mkdir(exist_ok=True)

fact = pd.read_csv(FACT, parse_dates=["txn_date"])
print(f"{len(fact):,} rows, {fact.shape[1]} columns")

260,287 rows, 46 columns


## Slim fact

GRAIN: one row per payment attempt. Unchanged from Task 4 - only the column
list is reduced. `ticket_band` is materialised here rather than computed in DAX
so that every visual bands identically.

In [8]:
TICKET_EDGES = [-np.inf, 25, 50, 100, 250, 500, np.inf]
TICKET_LABELS = ["under $25", "$25-50", "$50-100", "$100-250", "$250-500", "over $500"]
BAND_ORDER = {b: i for i, b in enumerate(TICKET_LABELS, start=1)}

f = fact.copy()
f["ticket_band"] = pd.cut(f.gross_usd, TICKET_EDGES, labels=TICKET_LABELS).astype(str)
f.loc[~f.is_success, "ticket_band"] = "n/a"

for col in ["merchant_category", "merchant_country", "merchant_risk_band",
            "merchant_pricing_plan", "customer_segment", "channel", "provider",
            "merchant_id", "customer_id"]:
    f[col] = f[col].fillna("(unknown)")

FACT_COLS = [
    "transaction_id", "txn_date", "txn_month",
    "merchant_id", "customer_id", "route_id", "provider",
    "channel", "status", "currency", "ticket_band",
    "merchant_category", "merchant_country", "merchant_pricing_plan",
    "merchant_risk_tier", "merchant_risk_band", "customer_segment",
    "is_attempt", "is_success", "is_reversal",
    "gross_usd", "gross_usd_constant_fx",
    "merchant_revenue_usd", "processing_cost_usd", "chargeback_cost_usd",
    "contribution_usd", "fx_impact_usd",
    "total_processing_ms", "max_risk_score", "model_version",
    "chargeback_count", "revenue_imputed",
    "dq_missing_merchant", "dq_quarantined",
]
fact_pbi = f[FACT_COLS].copy()
fact_pbi["is_attempt"] = fact_pbi.is_attempt.astype(int)
fact_pbi["is_success"] = fact_pbi.is_success.astype(int)

fact_pbi.to_csv(OUT / "fact_payment.csv", index=False)
print(f"fact_payment.csv  {len(fact_pbi):,} rows x {fact_pbi.shape[1]} cols  "
      f"{(OUT / 'fact_payment.csv').stat().st_size / 1e6:.1f} MB")

fact_payment.csv  260,287 rows x 34 cols  71.6 MB


## Dimensions

In [9]:
dim_date = pd.DataFrame({"date_key": pd.date_range("2025-01-01", "2025-12-31", freq="D")})
dim_date["month_key"] = dim_date.date_key.dt.to_period("M").astype(str)
dim_date["month_name"] = dim_date.date_key.dt.strftime("%b %Y")
dim_date["month_number"] = dim_date.date_key.dt.month
dim_date["quarter"] = dim_date.date_key.dt.to_period("Q").astype(str)
dim_date["year"] = dim_date.date_key.dt.year

dim_ticket = pd.DataFrame({"ticket_band": TICKET_LABELS + ["n/a"]})
dim_ticket["sort_order"] = dim_ticket.ticket_band.map(BAND_ORDER).fillna(99).astype(int)
dim_ticket["band_type"] = np.where(dim_ticket.ticket_band == "n/a", "Not applicable",
                          np.where(dim_ticket.sort_order <= 2, "Small ticket",
                          np.where(dim_ticket.sort_order <= 4, "Mid ticket", "Large ticket")))

dims = {
    "dim_date": dim_date,
    "dim_ticket_band": dim_ticket,
    "dim_merchant": pd.read_csv(DATA_DIR / "merchant.csv"),
    "dim_customer": pd.read_csv(DATA_DIR / "customer.csv"),
    "dim_route": pd.read_csv(DATA_DIR / "route_cost.csv")[["route_id", "provider", "region"]]
                   .drop_duplicates().reset_index(drop=True),
    "dim_pricing_plan": pd.read_csv(DATA_DIR / "pricing_plan.csv"),
}
# every dimension needs an (unknown) member so the fact never loses rows
dims["dim_merchant"].loc[len(dims["dim_merchant"])] = \
    ["(unknown)", "(unknown)", "(unknown)", "(unknown)", "(unknown)"]

for name, df in dims.items():
    df.to_csv(OUT / f"{name}.csv", index=False)
    print(f"{name:<20} {len(df):>7,} rows")

dim_date                 365 rows
dim_ticket_band            7 rows
dim_merchant             601 rows
dim_customer          15,000 rows
dim_route                  9 rows
dim_pricing_plan           4 rows


## Analysis tables

Two things DAX cannot reproduce cleanly, so they are computed here and loaded
as tables. Both are disconnected from the star - they are reported, not sliced.

In [10]:
s = f[(f.is_success == 1) & (~f.dq_quarantined)].copy()
FIRST, LAST = s.txn_month.min(), s.txn_month.max()

DIMS = [("ticket_band", "Ticket size band"),
        ("merchant_category", "Merchant category"),
        ("merchant_country", "Merchant country"),
        ("provider", "Provider"),
        ("channel", "Payment channel"),
        ("customer_segment", "Customer segment"),
        ("merchant_pricing_plan", "Pricing plan"),
        ("merchant_risk_band", "Merchant risk band")]

rows = []
for dim, label in DIMS:
    a, b = s[s.txn_month == FIRST], s[s.txn_month == LAST]
    idx = sorted(set(a[dim]) | set(b[dim]))
    w0 = a[dim].value_counts(normalize=True).reindex(idx).fillna(0)
    w1 = b[dim].value_counts(normalize=True).reindex(idx).fillna(0)
    c0 = a.groupby(dim).contribution_usd.mean().reindex(idx).fillna(0)
    c1 = b.groupby(dim).contribution_usd.mean().reindex(idx).fillna(0)
    mix, within = (w1 - w0) * c0, w1 * (c1 - c0)
    for seg in idx:
        rows.append((label, dim, seg, round(w0[seg], 4), round(w1[seg], 4),
                     round(c0[seg], 4), round(c1[seg], 4),
                     round(mix[seg], 4), round(within[seg], 4),
                     round(mix[seg] + within[seg], 4)))

decomp = pd.DataFrame(rows, columns=[
    "dimension", "column_name", "segment", "share_open", "share_close",
    "contrib_per_txn_open", "contrib_per_txn_close",
    "mix_effect", "within_effect", "total_effect"])
total = decomp.groupby("dimension").total_effect.transform("sum")
decomp["pct_of_total_change"] = (100 * decomp.total_effect / total).round(1)
decomp.to_csv(OUT / "analysis_decomposition.csv", index=False)
print("analysis_decomposition.csv", len(decomp), "rows")

bridge = pd.DataFrame([
    ("1. Contribution per txn, opening month", "Start",
     round(s[s.txn_month == FIRST].contribution_usd.mean(), 4)),
    ("2. Merchant revenue per txn", "Commercial",
     round(s[s.txn_month == LAST].merchant_revenue_usd.mean()
           - s[s.txn_month == FIRST].merchant_revenue_usd.mean(), 4)),
    ("3. Processing cost per txn", "Operational",
     round(-(s[s.txn_month == LAST].processing_cost_usd.mean()
             - s[s.txn_month == FIRST].processing_cost_usd.mean()), 4)),
    ("4. Chargeback cost per txn", "Risk",
     round(-(s[s.txn_month == LAST].chargeback_cost_usd.mean()
             - s[s.txn_month == FIRST].chargeback_cost_usd.mean()), 4)),
    ("5. Contribution per txn, closing month", "End",
     round(s[s.txn_month == LAST].contribution_usd.mean(), 4)),
], columns=["step", "category", "value_usd"])
bridge.to_csv(OUT / "analysis_bridge.csv", index=False)
print("analysis_bridge.csv")
bridge

analysis_decomposition.csv 35 rows
analysis_bridge.csv


,step,category,value_usd
0,"1. Contribution per txn, opening month",Start,1.6132
1,2. Merchant revenue per txn,Commercial,-0.7437
2,3. Processing cost per txn,Operational,0.1090
3,4. Chargeback cost per txn,Risk,0.1020
4,"5. Contribution per txn, closing month",End,1.0805


## Measure reference for the report

In [11]:
measures = pd.DataFrame([
 ("Attempted Volume", "SUM(fact_payment[is_attempt])", "Additive", "Count of payment attempts"),
 ("Successful Volume", "SUM(fact_payment[is_success])", "Additive", "Count of captured transactions"),
 ("Success Rate", "DIVIDE([Successful Volume], [Attempted Volume])", "NOT additive", "Ratio"),
 ("Gross Payment Value", "SUM(fact_payment[gross_usd])", "Additive", "USD at settlement FX rate"),
 ("Merchant Revenue", "SUM(fact_payment[merchant_revenue_usd])", "Additive", "Fee charged to merchant"),
 ("Processing Cost", "SUM(fact_payment[processing_cost_usd])", "Additive", "Provider cost, dated fee"),
 ("Chargeback Cost", "SUM(fact_payment[chargeback_cost_usd])", "Additive", "Disputed value plus handling fee"),
 ("Contribution Profit", "[Merchant Revenue] - [Processing Cost] - [Chargeback Cost]", "Additive", "USD"),
 ("Contribution per Successful Txn",
  "DIVIDE([Contribution Profit], [Successful Volume])", "NOT additive", "THE EXECUTIVE KPI"),
 ("Effective Take Rate", "DIVIDE([Merchant Revenue], [Gross Payment Value])", "NOT additive",
  "Compare within pricing plan only"),
 ("Cost per Successful Txn", "DIVIDE([Processing Cost], [Successful Volume])", "NOT additive", "Ratio"),
 ("Chargeback Rate", "DIVIDE(SUM(fact_payment[chargeback_count]), [Successful Volume])",
  "NOT additive", "Ratio"),
], columns=["Measure", "DAX", "Additive?", "Note"])
measures.to_csv(OUT / "measure_reference.csv", index=False)
measures

,Measure,DAX,Additive?,Note
0,Attempted Volume,SUM(fact_payment[is_attempt]),Additive,Count of payment attempts
1,Successful Volume,SUM(fact_payment[is_success]),Additive,Count of captured transactions
2,Success Rate,"DIVIDE([Successful Volume], [Attempted Volume])",NOT additive,Ratio
3,Gross Payment Value,SUM(fact_payment[gross_usd]),Additive,USD at settlement FX rate
4,Merchant Revenue,SUM(fact_payment[merchant_revenue_usd]),Additive,Fee charged to merchant
5,Processing Cost,SUM(fact_payment[processing_cost_usd]),Additive,"Provider cost, dated fee"
6,Chargeback Cost,SUM(fact_payment[chargeback_cost_usd]),Additive,Disputed value plus handling fee
7,Contribution Profit,[Merchant Revenue] - [Processing Cost] - [Char...,Additive,USD
8,Contribution per Successful Txn,"DIVIDE([Contribution Profit], [Successful Volu...",NOT additive,THE EXECUTIVE KPI
9,Effective Take Rate,"DIVIDE([Merchant Revenue], [Gross Payment Value])",NOT additive,Compare within pricing plan only


In [12]:
print("Files written to", OUT.resolve())
for p in sorted(OUT.glob("*.csv")):
    print(f"  {p.name:<32} {p.stat().st_size / 1e6:>7.2f} MB")

Files written to C:\Users\Administrator\Downloads\astrapay-capstone\07_powerBI\powerbi_model
  analysis_bridge.csv                 0.00 MB
  analysis_decomposition.csv          0.00 MB
  dim_customer.csv                    0.60 MB
  dim_date.csv                        0.02 MB
  dim_merchant.csv                    0.02 MB
  dim_pricing_plan.csv                0.00 MB
  dim_route.csv                       0.00 MB
  dim_ticket_band.csv                 0.00 MB
  fact_payment.csv                   71.62 MB
  measure_reference.csv               0.00 MB
